# LULC

This notebook does 3 things:
1. Generate training data for LULC classification. It uses 3 existing LULC products.
2. Filter training data per class using kmeans clustering to exclude outliers.
3. Train a random forest model, and then predict LULC.

### Steps: 
1. Training data:
For each site extract training data.
  -> Find product agreement. mask to gadm 100m buffer.
  -> Add geomad, indices, elevation etc.
  -> Write csv per site of training data.
do this in a notebook. don't commit training data.

1a. Filter outliers per class.

2. Train model:
Try one model for all sites. (append all CSVs)
export model dump (python pickle?)
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

3. Predict:
per grid tile:
  per year:
    load geomad/indices/elevation etc.
    make using get_gadm (buffered 100m)
    predict
This is as a command. 

### Load packages


In [17]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
%matplotlib inline

# Scientific core
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.gridspec as gridspec
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

# Geospatial
import geopandas as gpd
import rioxarray # noqa: F401
import xarray as xr
import zarr # noqa: F401
from ipyleaflet import basemaps
from odc.geo.xr import assign_crs
from odc.stac import load
from planetary_computer import sign_url
from pystac.client import Client
from shapely.geometry import box

# Machine learning
from scipy.ndimage import minimum_filter
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Local
from ldn.grids import get_gadm
from ldn.random_sampling import random_sampling
from ldn.typology import classes, colors, world_cover_map, cci_lc_map, io_map, lvl1
from ldn.utils import get_geomad_dem_indices
from notebooks.src.Compare_LULC_func import standardise_class

In [ ]:
# Parameters
datetime_year = "2020"

wgs84 = 'EPSG:4326' # WGS84
class_attr='lulc'
buffer_m  = 100
# Use Planetary Computer STAC API
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")


"""
# For Singapore:
country_of_interest = {"Singapore": "SGP"}
analysis_crs = 'EPSG:6933' # Equal Earth for global analysis
stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ci_ls_geomad/0-0-2/ci_ls_geomad.parquet" # Non-Pacific
"""
# For Fiji:
country_of_interest = {"Fiji": "FJI"}
analysis_crs = 'EPSG:3832' # PDC Mercator for antimeridian avoiding
stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_geomad/0-0-2/ausp_ls_geomad.parquet" # Pacific


country_name = list(country_of_interest.keys())[0]
country_gadm = get_gadm(countries=country_of_interest, overwrite=True)

# This is just for Fiji. We are subsetting it to about half the country bounds so it doesn't cross the antimeridian.
# Don't do this for other countries!
print(country_gadm.total_bounds)
country_gadm = country_gadm.clip(box(177.0, -18.5, 178.5, -17.5)) # Small box for developing
# country_gadm = country_gadm.clip(box(0, -22, 179.5, -13)) # Big box for production
print(country_gadm.total_bounds)
country_gadm.explore()

In [ ]:
# Buffer country polygon to include coastal zones.
# Fiji and Singapore are both a single multipolygon from GADM.
country_wgs84_buffered = gpd.GeoDataFrame(
    geometry=country_gadm.to_crs(analysis_crs).buffer(buffer_m).to_crs(wgs84),
    crs=wgs84
)
country_wgs84_buffered.explore()

In [21]:
# Takes ~9 min for Fiji subset.
# Takes ~5 min for Fiji subset (small).
geomad_dem = get_geomad_dem_indices(country_wgs84_buffered, stac_geoparquet, datetime_year, catalog=catalog)

# Singapore and Fiji only have 2 GeoMAD tiles for 2020 as of 2026-03-05. GeoMAD will be fully processed in future.
# Elevation has the full extent of Fiji. GeoMedian/GeoMAD only have the processed tile extents.

(177.0285359770697, -18.50085705474703, 178.5008983152841, -17.499138007537496)
Found 1 GeoMAD items for this region and year
Available bands (excluding count): ['green', 'swir16', 'smad', 'emad', 'red', 'bcmad', 'nir08', 'blue', 'swir22']
GeoMAD dataset loaded CRS (should be native): 3832
GeoMAD bands loaded: ['green', 'swir16', 'smad', 'emad', 'red', 'bcmad', 'nir08', 'blue', 'swir22']
GeoMAD dataset shape: FrozenMappingWarningOnValuesAccess({'y': 3886, 'x': 5464})
Found 4 DEM items for this AOI


In [22]:
# Write data as Zarr. Just read this to skip having to rerun the above code while developing the rest of the notebook.
geomad_dem.to_zarr(f"test_geomad_dem_{country_name}.zarr", mode="w")

In [ ]:
# This cell is just to check the data visually.
for column in geomad_dem.data_vars:
    # print(f"{column}: {country_geomad_dem[column].shape}, {country_geomad_dem[column].dtype}")
    print(f"Column: {column}, min: {geomad_dem[column].min().values}, max: {geomad_dem[column].max().values}, dtype: {geomad_dem[column].dtype}")

# test_attr = "slope"
# test_attr = "elevation"
# test_attr = "aspect"
test_attr = "ndvi"
# test_attr = "red"
test_attr_colormap = {"elevation": "terrain", "aspect": "viridis", "slope": "plasma", "ndvi": "YlGn", "red": "Reds"}[test_attr]

geomad_dem[test_attr].odc.explore(
    cmap=test_attr_colormap,
    vmin=geomad_dem[test_attr].min().values,
    vmax=geomad_dem[test_attr].max().values,
)

## Load 3 LU/LC products for country.

In [24]:
# Steps:
# 1. Load the highest res product in 4326 (there are 2 at 10m.)
# 2. Load the others to be like the 1st. Use nearest resampling to avoid creating new classes that may not exist in the original data.
# 3. Find agreement where 2 out of 3 of the products match exactly. CCI is probably going to agree the least due to its different resolution. Neighbours also agree.

# A generic function to load a product given the boundary
def load_lulc_data(aoi_4326: gpd.GeoDataFrame, product: str, like: xr.Dataset | None) -> xr.Dataset:
    print("Search/Load bounds: ", list(aoi_4326.total_bounds))
    assert datetime_year == "2020", "Year must be '2020'"
    lulc_items = catalog.search(
        collections=[product],
        bbox=list(aoi_4326.total_bounds),
        datetime=f"{datetime_year}-01-01/{datetime_year}-12-31",
    )
    lulc_items = list(lulc_items.items())

    # Filter IO wrong year data
    if "output_band" == "io":
        lulc_items = [i for i in lulc_items if datetime_year in i.id]
    print(f"Found {len(lulc_items)} {product} LULC STAC items for this AOI")

    for item in lulc_items:
        print(f"  {item.id} ({item.bbox})")

    if like is not None:
        # IO tiles (and potentially others) are in UTM with world-spanning bboxes.
        # Load in native CRS first using geopolygon to constrain extent, then reproject.

        # Loading IO "like" WC using ODC STAC load was meaning all data was missing. Have to load and then reproject like WC.

        ds = load(
            lulc_items,
            geopolygon=aoi_4326,
            chunks={},
            patch_url=sign_url,
        )
        print(f"Native CRS: {ds.odc.crs.epsg}, shape: {ds.dims}")
        print(f"Native unique values: {np.unique(ds.isel(time=0).to_array().values)}")

        ds = ds.odc.reproject(like.odc.geobox, resampling="nearest")
        print(f"Reprojected CRS: {ds.odc.crs.epsg}, shape: {ds.dims}")
    else:
        ds = load(
            lulc_items,
            geopolygon=aoi_4326,
            crs=wgs84,
            chunks={},
            patch_url=sign_url,
        )

    print(f"Product {product} has variables: {ds.data_vars}")
    return ds

In [25]:
LULC_PRODUCTS = [
    {
        "product":     "esa-worldcover", # 10m
        "native_band": "map",
        "output_band": "esa_wc",
        "class_map":   world_cover_map,
        # Retain pixels with at least 1 valid Sentinel-1/2 observation in each of the 3 seasons.
        # TODO: Relax this to 2/3. It filters a lot of Fiji.
        "quality_fn":  lambda ds: (ds["input_quality.1"] > 0) & (ds["input_quality.2"] > 0) & (ds["input_quality.3"] > 0),
        "quality_bands": ["input_quality.1", "input_quality.2", "input_quality.3"],
    },
    {
        "product":     "esa-cci-lc", # 300m
        "native_band": "lccs_class",
        "output_band": "esa_cci",
        "class_map":   cci_lc_map,
        # Retain pixels that were processed, had a stable class across the time series, and had at least 3 observations.
        # TODO: Check how much this filters.
        "quality_fn":  lambda ds: (ds["processed_flag"] == 1) & (ds["change_count"] == 0) & (ds["observation_count"] >= 3),
        "quality_bands": ["processed_flag", "change_count", "observation_count"],
    },
    {
        "product":     "io-lulc-annual-v02", # 10m
        "native_band": "data",
        "output_band": "io",
        "class_map":   io_map,
        # No quality band available
        "quality_fn":  None,
        "quality_bands": [],
    },
]


def load_and_prepare(aoi_4326: gpd.GeoDataFrame, product_dict: dict, like: xr.Dataset | None) -> xr.Dataset:
    product, native_band, output_band, class_map, quality_fn, quality_bands = product_dict.values()
    ds = load_lulc_data(aoi_4326, product, like=like)
    ds[output_band] = ds[native_band]

    # Keep quality bands until after masking
    bands_to_keep = {output_band} | set(quality_bands)
    ds = ds.drop_vars([v for v in ds.data_vars if v not in bands_to_keep])
    ds = ds.isel(time=0)

    # Set spatial dims and CRS while still lazy
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    ds = ds.rio.write_crs(wgs84, inplace=True)

    # Clip and quality mask before .load() to avoid materialising full tiles
    # Precise clip to actual AOI geometry
    # Clip in 4326 because products are loading in that CRS.
    ds = ds.rio.clip(aoi_4326.geometry.values, crs=wgs84, from_disk=True, all_touched=True)

    # TODO: Reenable quality masking.
    # # Quality mask
    # if quality_fn is not None:
    #     ds[output_band] = ds[output_band].where(quality_fn(ds))

    # Drop quality bands then load only the clipped, masked result
    ds = ds.drop_vars([v for v in ds.data_vars if v != output_band])
    ds = ds.load()
    ds[output_band] = ds[output_band].astype("int16") # Ensure integer type for class labels

    print(f"Values before standardisation (dtype: {ds[output_band].dtype}):")
    print(f"Unique values: {np.unique(ds[output_band].values)}")

    ds[output_band] = standardise_class(ds[output_band], class_map)
    return ds


# Load worldcover first (highest native res), then align others to it
reference, *rest = LULC_PRODUCTS

wc_10m = load_and_prepare(country_wgs84_buffered, reference, like=None)

other_products = [
    load_and_prepare(country_wgs84_buffered, p, like=wc_10m)
    for p in rest
]

cci_10m, io_10m = other_products

Search/Load bounds:  [np.float64(177.0285359770697), np.float64(-18.50085705474703), np.float64(178.5008983152841), np.float64(-17.499138007537496)]
Found 2 esa-worldcover LULC STAC items for this AOI
  ESA_WorldCover_10m_2020_v100_S21E177 ([177.0, -19.49125, 179.9999167, -18.0])
  ESA_WorldCover_10m_2020_v100_S18E177 ([177.0, -18.0, 179.9999167, -15.8490833])
Product esa-worldcover has variables: Data variables:
    map              (time, latitude, longitude) uint8 212MB dask.array<chunksize=(1, 12022, 17669), meta=np.ndarray>
    input_quality.1  (time, latitude, longitude) int16 425MB dask.array<chunksize=(1, 12022, 17669), meta=np.ndarray>
    input_quality.2  (time, latitude, longitude) int16 425MB dask.array<chunksize=(1, 12022, 17669), meta=np.ndarray>
    input_quality.3  (time, latitude, longitude) int16 425MB dask.array<chunksize=(1, 12022, 17669), meta=np.ndarray>
Values before standardisation (dtype: int16):
Unique values: [ 0 10 20 30 40 50 60 80 90 95]
Search/Load bounds

In [26]:

# Write products data as Zarr. Just read this to skip having to rerun the above code while developing the rest of the notebook.
wc_10m.to_zarr(f"test_product_{country_name}_wc.zarr", mode="w")
cci_10m.to_zarr(f"test_product_{country_name}_cci.zarr", mode="w")
io_10m.to_zarr(f"test_product_{country_name}_io.zarr", mode="w")

"""
# Read for faster development.
wc_10m = xr.open_zarr(f"test_product_{country_name}_wc.zarr", consolidated=True)
cci_10m = xr.open_zarr(f"test_product_{country_name}_cci.zarr", consolidated=True)
io_10m = xr.open_zarr(f"test_product_{country_name}_io.zarr", consolidated=True)
"""

'\n# Read for faster development.\nwc_10m = xr.open_zarr(f"test_product_{country_name}_wc.zarr", consolidated=True)\ncci_10m = xr.open_zarr(f"test_product_{country_name}_cci.zarr", consolidated=True)\nio_10m = xr.open_zarr(f"test_product_{country_name}_io.zarr", consolidated=True)\n'

In [27]:
# Print class distributions before agreement analysis.
print ("Class distribution:")
for name, ds, band in [
    ("wc", wc_10m, 'esa_wc'),
    ("cci", cci_10m, 'esa_cci'),
    ("io", io_10m, 'io'),
]:
    vals, counts = np.unique(ds[band].values, return_counts=True)
    print(name)
    print(pd.DataFrame({"class": vals, "count": counts}))

# CCI has no Other (7) class in Singapore or Fiji.
# Fiji has a lot of nodata (0) because of vast oceas between islands. This is expected.

# For Fiji IO is loading with all class 0.
# TODO: Fix this. It is a bg somewhere.


Class distribution:
wc
   class     count
0      0  97226295
1      1  92170043
2      2  19283115
3      3    414931
4      4   1118757
5      5    540234
6      6   1252822
7      7    410521
cci
   class     count
0      0  97226295
1      1  68751666
2      2   7938053
3      3  34403832
4      4   2709039
5      5    362090
6      6   1025743
io
   class     count
0      0  97226338
1      1  89527720
2      2  14947100
3      3   5643238
4      4     58497
5      5   3145542
6      6   1817925
7      7     50358


# Create layer where all 2 out of 3 land cover products agree.
Each pixel and its 8 neighbours must have agreement for 2 out of the 3 products.

In [28]:
print("Finding agreement:")

# Count pairwise agreements at each pixel (0, 1, 2, or 3 pairs agree)
wc  = wc_10m['esa_wc']
# Snap to WC grid. Needed. Maybe caused by clipping?
cci = cci_10m['esa_cci'].reindex_like(wc, method="nearest", fill_value=0)
io  = io_10m['io'].reindex_like(wc, method="nearest", fill_value=0)

# Valid pixels — all three products have data for this region.
valid = (wc > 0) & (cci > 0) & (io > 0)

pair_agree_count = (
    (wc == cci).astype("int8") +
    (cci == io).astype("int8") +
    (wc == io).astype("int8")
)

# At least 2 of 3 products agree — take the majority class
majority_class = xr.where(wc == cci, wc, io)  # if wc==cci, that's the majority; else io agrees with one of them
two_of_three = (pair_agree_count >= 2) & valid

# All 8 neighbours must also have 2/3 agreement
neighbour_mask = xr.DataArray(
    minimum_filter(two_of_three.values.astype("float32"), size=3, mode="constant", cval=0) == 1,
    coords=two_of_three.coords, dims=two_of_three.dims,
)

agreed_class = majority_class.where(neighbour_mask & two_of_three, other=0).rename(class_attr)

# Diagnostics
label_map = {c["value"]: c["label"] for c in lvl1.values()}
_classes, counts = np.unique(agreed_class.values, return_counts=True)
counts_df = pd.DataFrame({"class": _classes, "pixel_count": counts})
counts_df["label"] = counts_df["class"].map(label_map)
print(counts_df)

# Write agreeing pixels as tiff.
agreed_class.odc.write_cog(f'lulc_agree_{country_name}.tif', overwrite=True)

Finding agreement:
   class  pixel_count       label
0      0    146211567     No data
1      1     64498568  Tree Cover
2      2       884467   Grassland
3      3       170351    Cropland
4      4        32738     Wetland
5      5        92700    Built-up
6      6       526327       Water


PosixPath('lulc_agree_Fiji.tif')

In [ ]:
"""
# Load the agreeing pixels back in to check they look correct.
agreed_class = rioxarray.open_rasterio(f'lulc_agree_{country_name}.tif', dtype="int16")
"""

# Visualise the agreeing classes from 3 products

In [ ]:
with plt.ioff():  # prevents double-rendering in some notebook backends
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    vals, counts = np.unique(agreed_class.values, return_counts=True)
    print(pd.DataFrame({"class": vals, "count": counts}))

    cmap = ListedColormap([colors[k] for k in vals])
    norm = BoundaryNorm(list(vals) + [np.max(vals) + 1], cmap.N)
    agreed_class.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_labels=True, add_colorbar=False, interpolation='none')
    patches_list = [Patch(facecolor=colors[k]) for k in colors]
    ax.legend(patches_list, list(classes.keys()), loc='upper center', ncol=4, bbox_to_anchor=(0.5, -0.1))
    ax.set_title("Agreed LULC class")

    display(fig)
    plt.close(fig)

## Generate random training samples
We generate some randomly distributed samples for each class from the clipped classification map using the `random_sampling` function. This function takes in a few parameters:  
* `da`: a classified map in the format of 2-dimensional xarray.DataArray
* `n`: total number of points to sample.
* `min_sample_n`: Minimum number of samples to generate per class if proportional number is smaller than this. **Note that the resultant number of samples may be higher than the set `n` due to setting of this minimum number of samples.** 
* `sampling`: the sampling strategy, e.g. 'stratified_random' where each class has a number of points proportional to its relative area, 'equal_stratified_random' where each class has the same number of points, or 'manual' which allows you to define number of samples for each class.
* `out_fname`: a filepath name for the function to export a shapefile/geojson of the sampling points into a file. You can set this to `None` if you don't need to output the file.
* `class_attr`: This is the column name of output dataframe that contains the integer class values on the classified map. 
* `drop_value`: pixel value on the classification map to be excluded from sampling.

The output of the function is a geopandas dataframe of randomly distributed points containing a column `class_attr` identifying class values. 

Here we extract around 1000 training points in total and export the points in a geojson file for use in the rest of workflow. Here we use the stratified sampling method by setting 'equal_stratified_random', but also set the minimum number of samples as 75 to avoid missing samples for some minor classes. 

As mentioned earlier we don't want the abandoned classes to be included in the samples we set drop_value as 0 before implementing the function:

In [43]:
n=2100
min_sample_n=300
drop_value=0

print("Sampling training points from agreed product pixels:")
samples = (
    random_sampling(
        da=agreed_class,
        n=n,
        sampling='stratified_random',
        min_sample_n=min_sample_n,
        out_fname=f"{country_name}_samples.geojson", # None
        class_attr=class_attr,
        drop_value=drop_value,
    )
)

Sampling training points from agreed product pixels:
      proportion  n_available  class
lulc                                
1       0.974223     64498568      1
2       0.013359       884467      2
6       0.007950       526327      6
3       0.002573       170351      3
5       0.001400        92700      5
4       0.000494        32738      4
Class 1: sampling at 2046locations
Class 2: sampling at 300locations
Class 6: sampling at 300locations
Class 3: sampling at 300locations
Class 5: sampling at 300locations
Class 4: sampling at 300locations


In [44]:
samples.head()

,lulc,geometry
0,1,POINT (177.72846 -18.17662)
1,1,POINT (177.73021 -17.68779)
2,1,POINT (178.32621 -18.10262)
3,1,POINT (177.63038 -17.91671)
4,1,POINT (178.00113 -17.71171)


## Visualise the training data by class

In [ ]:
samples.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(samples[class_attr].unique())), # Walrus operator to assign and pass the value in one step.
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

## Extract Geomedian/GeoMAD values at training data point locations.

For now we run and load individual tiles and mosaic them because we have not run all tiles yet.

Once we have finalised the GeoMAD data, we will set up a GeoParquet STAC index for it. Then querying it by bbox will be simple and replace the more complicated process implemented here.

In [46]:

print("GeoMAD DEM bands:")
band_names_with_indices = [v for v in geomad_dem.data_vars]
print(band_names_with_indices)

# Extract Geomedian and GeoMAD elevation, indices values at each sample point location

# Reproject sample points to match native CRS of geomad before sampling.
gpd_random_samples_analysis = samples.to_crs(analysis_crs)

# Build point-wise x/y arrays with the SAME 'points' index
points_idx = np.arange(len(gpd_random_samples_analysis))
xs = xr.DataArray(gpd_random_samples_analysis.geometry.x.values, dims="points", coords={"points": points_idx})
ys = xr.DataArray(gpd_random_samples_analysis.geometry.y.values, dims="points", coords={"points": points_idx})
# Point-wise nearest sampling (one sampled pixel per input point)
sampled = geomad_dem[band_names_with_indices].sel(x=xs, y=ys, method="nearest")

# Convert to dataframe indexed by points; keep exactly one row per point
sampled_df = sampled.to_dataframe().reset_index()
sampled_df = sampled_df.sort_values("points").reset_index(drop=True)

# Merge back to training points (original CRS)
samples = samples.reset_index(drop=True).copy()
samples = pd.concat([samples, sampled_df[band_names_with_indices]], axis=1)

print(f"Training data shape: {samples.shape}")
print("Unique values per band:")
print(samples[band_names_with_indices].nunique())
samples.head()

GeoMAD DEM bands:
['green', 'swir16', 'smad', 'emad', 'red', 'bcmad', 'nir08', 'blue', 'swir22', 'ndvi', 'ndwi', 'mndwi', 'ndti', 'bsi', 'mbi', 'baei', 'bui', 'elevation', 'slope', 'aspect']
Training data shape: (3546, 22)
Unique values per band:
green        1315
swir16       1584
smad         1780
emad         1794
red          1291
bcmad        1794
nir08        1669
blue         1181
swir22       1400
ndvi         1793
ndwi         1794
mndwi        1794
ndti         1791
bsi          1794
mbi          1794
baei         1794
bui          1794
elevation    3243
slope        3322
aspect       3299
dtype: int64


,lulc,geometry,green,swir16,smad,emad,red,bcmad,nir08,blue,...,ndwi,mndwi,ndti,bsi,mbi,baei,bui,elevation,slope,aspect
0,1,POINT (177.72846 -18.17662),0.024070,0.089245,0.001527,3665.285400,0.013483,0.047109,0.262522,0.007598,...,-0.832026,-0.575167,-0.281939,-0.448957,-0.036417,2.766470,-1.394893,252.864151,15.021321,84.000816
1,1,POINT (177.73021 -17.68779),0.023960,0.081518,0.001424,1663.602417,0.015572,0.022518,0.206505,0.009660,...,-0.792073,-0.545685,-0.212167,-0.380122,0.005511,2.991847,-1.293707,552.884155,23.455675,87.446503
2,1,POINT (178.32621 -18.10262),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.802742,10.379505,285.875031
3,1,POINT (177.63038 -17.91671),0.032705,0.104507,0.000257,1778.767456,0.018735,0.028620,0.305862,0.011970,...,-0.806804,-0.523294,-0.271579,-0.441172,-0.037590,2.322930,-1.375232,446.719513,16.968615,77.702324
4,1,POINT (178.00113 -17.71171),0.062047,0.147352,0.000613,3107.007568,0.038343,0.028514,0.374750,0.027342,...,-0.715898,-0.407378,-0.236129,-0.368156,0.003554,1.615771,-1.249905,776.830872,7.924020,282.967407


## Visualise GeoMedian red values per sample point

In [ ]:
# Visualise red values using the actual data range
samples.explore(
    column="red",
    cmap="Reds",
    k=7,
    scheme="natural_breaks",
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

# For Fiji this looks bad because GeoMAD tiles have not been run fully for 2020 yet.

## Explore elevation data on training data points

In [ ]:
samples.explore(
    column="elevation",
    cmap="Blues",
    scheme="natural_breaks",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore NDVI data on training data points

In [ ]:
samples.explore(
    column="ndvi",
    cmap="RdYlGn",
    scheme="quantiles",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

In [ ]:
# TODO: Run GeoMAD for 2020 for Fiji so the next steps work. Right now about half the points are missing this data.

# Section 2: Filter training data by K-Means clustering
Goal is to reduce confusion. Make training point classes clean/accurate/spectrally separable.

In [ ]:
# 1. Prep feature matrix ────────────────────────────────────────────────────
exclude_cols = ['lulc', 'geometry']
feature_cols = [c for c in samples.columns if c not in exclude_cols]

samples['outlier'] = False
samples.loc[samples[feature_cols].isna().any(axis=1), 'outlier'] = True

nan_rows = samples['outlier'].sum()
print(f"Rows with NaNs (auto-flagged): {nan_rows}")

valid_mask = ~samples[feature_cols].isna().any(axis=1)
valid_idx  = samples.index[valid_mask]

X = samples.loc[valid_idx, feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Visualisation helpers ──────────────────────────────────────────────────
CLUSTER_CMAP = 'tab10'

def plot_pca(ax, Xc, labels, centers, outlier_mask, class_name, best_score):
    """PCA scatter — inliers coloured by cluster, outliers as red crosses."""
    pca      = PCA(n_components=2)
    X_2d     = pca.fit_transform(Xc)
    c_2d     = pca.transform(centers)
    var      = pca.explained_variance_ratio_
    k        = len(np.unique(labels))
    inliers  = ~outlier_mask

    ax.scatter(
        X_2d[inliers, 0], X_2d[inliers, 1],
        c=labels[inliers], cmap=CLUSTER_CMAP, vmin=0, vmax=max(k - 1, 1),
        alpha=0.55, s=25, linewidths=0, label='inlier'
    )
    if outlier_mask.any():
        ax.scatter(
            X_2d[outlier_mask, 0], X_2d[outlier_mask, 1],
            c='red', marker='x', s=50, linewidths=1.2, label='outlier'
        )
    ax.scatter(
        c_2d[:, 0], c_2d[:, 1],
        c='black', marker='*', s=180, zorder=5, label='centroid'
    )
    ax.set(
        xlabel=f'PC1 ({var[0]:.1%})',
        ylabel=f'PC2 ({var[1]:.1%})',
        title=f'PCA  |  k={k}  sil={best_score:.3f}'
    )
    ax.legend(fontsize=7, markerscale=0.9)


def plot_heatmap(ax, centers, feature_cols, outlier_count, n):
    """Centroid heatmap — rows = clusters, cols = features."""
    im = ax.imshow(centers, aspect='auto', cmap='RdBu_r')
    ax.set_xticks(range(len(feature_cols)))
    ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(centers)))
    ax.set_yticklabels([f'C{i}' for i in range(len(centers))], fontsize=8)
    plt.colorbar(im, ax=ax, label='Scaled value', pad=0.02)

    # Annotate each cell with the scaled value
    for r in range(centers.shape[0]):
        for c in range(centers.shape[1]):
            ax.text(c, r, f'{centers[r, c]:.1f}',
                    ha='center', va='center', fontsize=6,
                    color='white' if abs(centers[r, c]) > 1.2 else 'black')

    ax.set_title(f'Centroids  |  outliers={outlier_count} ({100*outlier_count/n:.1f}%)')


# 3. Per-class clustering + outlier flagging + visualisation ────────────────
threshold = 2.0   # ← tune: lower = more aggressive flagging

_classes = samples.loc[valid_idx, 'lulc'].unique()

for cls in _classes:
    mask = (samples.loc[valid_idx, 'lulc'] == cls).values
    idx  = valid_idx[mask]
    Xc   = X_scaled[mask]
    n    = len(Xc)

    if n < 6:
        continue
    k_max = min(5, n // 5)
    if k_max < 2:
        continue

    # Optimal k via silhouette ──────────────────────────────────────────
    best_k, best_score = 2, -1
    for k in range(2, k_max + 1):
        km     = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xc)
        score  = silhouette_score(Xc, labels)
        if score > best_score:
            best_k, best_score = k, score

    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xc)
    labels   = km_final.labels_
    centers  = km_final.cluster_centers_

    # Outlier flagging ──────────────────────────────────────────────────
    outlier_mask = np.zeros(n, dtype=bool)
    for cluster_id in range(best_k):
        in_cluster = labels == cluster_id
        pts        = Xc[in_cluster]
        dists      = cdist(pts, centers[cluster_id].reshape(1, -1)).flatten()
        median_d   = np.median(dists)
        if median_d == 0:
            continue
        flagged = dists > threshold * median_d
        outlier_mask[np.where(in_cluster)[0][flagged]] = True

    samples.loc[idx[outlier_mask], 'outlier'] = True

    print(f"  {str(cls):30s} | n={n:4d} | k={best_k} | sil={best_score:.3f} "
        f"| outliers={outlier_mask.sum():4d} ({100*outlier_mask.mean():.1f}%)")

    # Figure: PCA (left) + heatmap (right) ─────────────────────────────
    fig = plt.figure(figsize=(14, 4.5))
    fig.suptitle(f'Class: {cls}  (n={n})', fontsize=12, fontweight='bold', y=1.01)

    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6], figure=fig)
    ax_pca  = fig.add_subplot(gs[0])
    ax_heat = fig.add_subplot(gs[1])

    plot_pca(ax_pca, Xc, labels, centers, outlier_mask, cls, best_score)
    plot_heatmap(ax_heat, centers, feature_cols, outlier_mask.sum(), n)

    plt.tight_layout()
    plt.show()

# 4. Summary ────────────────────────────────────────────────────────────────
print(f"\nTotal points : {len(samples):,}")
print(f"Clean        : {(~samples['outlier']).sum():,}")
print(f"Outliers     : {samples['outlier'].sum():,}")
print("Outliers per class:")
print(samples.groupby('lulc')['outlier'].mean().mul(100).round(1).astype(str) + '%')
samples = samples[~samples['outlier']]

In [ ]:
samples.drop(columns=["time", 'spatial_ref'], inplace=True)

out_fname = f"{country_name}_samples_2"

# Write the final training data
samples.to_file(f"{out_fname}.geojson", driver="GeoJSON", index=False)
samples.to_csv(f"{out_fname}.csv", index=False)

print(f"Saved training data as geojson and CSV to {out_fname}")

# Section 3: Train and Predict

In [ ]:
# Find and load training data
# training_data_file = f"{out_fname}.csv"
# samples = pd.read_csv(training_data_file)
# print(samples.columns)

samples.drop(columns=['outlier'], inplace=True)

# Split 70/30 into train/test. Splits the classes into train/test in a representative way.
train_gdf, test_gdf = train_test_split(samples, test_size=0.3, stratify=samples[class_attr], random_state=42)

print(f"Training set class distribution:\n{train_gdf[class_attr].value_counts()}")
print(f"Test set class distribution:\n{test_gdf[class_attr].value_counts()}")
print(train_gdf)

## Create a classifier and fit a model

_classes = train_gdf[class_attr].values.astype(int)
print(f"Classes: {np.unique(_classes)}")

# Observations — drop class and any remaining non-numeric columns
feature_cols = [c for c in train_gdf.columns
                if c != class_attr
                and pd.api.types.is_numeric_dtype(train_gdf[c])]
observations = train_gdf[feature_cols].values

# Create and fit
classifier = RandomForestClassifier(class_weight='balanced')
model = classifier.fit(observations, _classes)
# Define features and target

feature_cols = [c for c in train_gdf.columns if c != class_attr]

X_train = train_gdf[feature_cols].values
y_train = train_gdf[class_attr].values
X_test = test_gdf[feature_cols].values
y_test = test_gdf[class_attr].values

classifier = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Feature importance — drop noisy features
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)
# Feature importance is probably the most useful next step — it'll tell you which bands are actually helping and which are adding noise.

present = np.unique(np.concatenate([y_test, y_pred]))
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present]

print(classification_report(y_test, y_pred, target_names=target_names, labels=present))

stack = np.stack([geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
stack = np.stack([geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

predictions = model.predict(stack)

# Reshape back to raster
prediction_map = predictions.reshape(geomad_dem[feature_cols[0]].shape)

# Wrap in DataArray
predicted_da = xr.DataArray(
    prediction_map,
    coords={"y": geomad_dem.y, "x": geomad_dem.x},
    dims=["y", "x"],
    name="lulc",
)

## Visualise our results

predicted_da = assign_crs(predicted_da, crs=analysis_crs)

# Can't use this because there is no Other (7) class in prediction.
# data_classes = sorted(colors.keys())
# print(f"All classes: {data_classes}")
data_classes = [0, 1, 2, 3, 4, 5, 6]
cmap = ListedColormap([colors[c] for c in data_classes])

predicted_da.odc.explore(
    classes=data_classes,
    cmap=cmap,
    legend=True,
    tiles=basemaps.Esri.WorldImagery
)

# Aim for >80% accuracy. Don't just look at the confusion matrix, also look at the output map.

# Use a product for validation.
# One validation method for tuning and another for final measure.

# target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0]
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0 and v != 7] # Use this while there is no Other (7) class in the data.

# Classification report
print(classification_report(y_test, y_pred, target_names=target_names))
print(classification_report(y_test, y_pred, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(xticks_rotation=45, cmap="Blues")

predicted_da.odc.write_cog("predicted_lulc.tif", overwrite=True)